# 🧠 HUẤN LUYỆN MÔ HÌNH CUSTOMRESNET18 — NHẬN DIỆN 6 LOẠI CẢM XÚC (FROM SCRATCH)
**Dự án:** Nhận diện Cảm xúc Khuôn mặt phục vụ Nền tảng Phỏng vấn thông minh & Giám sát biểu cảm

**Quy trình 7 bước chuẩn Kỹ sư AI:**
1. 🟢 **Bước 1: Xác định bài toán & Tiêu chí đánh giá** (Define & Metrics)
2. 🟢 **Bước 2: Thu thập & Tiền xử lý dữ liệu** (Data Engineering & Exploration)
3. 🟢 **Bước 3: Tăng cường dữ liệu & Xử lý lệch pha** (Augmentation & Balancing)
4. 🟢 **Bước 4: Thiết lập Kiến trúc CustomResNet18 from scratch** (Architecture Setup)
5. 🟢 **Bước 5: Huấn luyện & Giám sát thực nghiệm** (Training & Monitoring)
6. 🟢 **Bước 6: Đánh giá độc lập & Ma trận nhầm lẫn** (Evaluation & Confusion Matrix)
7. 🟢 **Bước 7: Tối ưu hóa & Đóng gói triển khai** (Optimization & ONNX Export)

---
> **Thông số kỹ thuật cốt lõi:**
- **6 Nhãn cảm xúc:** `Contempt`, `Fear`, `Happy`, `Normal`, `Sad`, `Surprised`
- **Mô hình:** `CustomResNet18` tự xây dựng từ đầu (11.17M tham số, khởi tạo Kaiming He, không dùng pre-train)
- **Ảnh đầu vào:** Grayscale 48×48 (1 channel)
- **Mục tiêu hiệu suất:** Overall Validation Accuracy $\ge 70\%$, F1-Score $\ge 0.60$ mỗi lớp.


## ⚙️ Thiết lập Môi trường, Kết nối Google Drive & Kiểm tra GPU
1. Tự động kết nối Google Drive (nếu đang chạy trên Google Colab).
2. Kiểm tra phần cứng card đồ họa GPU (NVIDIA Tesla T4).
3. Cố định `seed` ngẫu nhiên để tái lập thực nghiệm (Reproducibility).


In [ ]:
# 1. Kết nối Google Drive nếu đang ở môi trường Google Colab
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Đã kết nối Google Drive thành công!')
except ImportError:
    IN_COLAB = False
    print('📌 Môi trường: Local Workspace (Máy tính cá nhân)')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Thiết lập thư viện & Hardware Check
import os
import sys
import time
import json
import random
import zipfile
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Thiết lập Random Seed cho tính tái lập thực nghiệm
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Kiểm tra phần cứng tính toán
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 65)
print(f'🖥️  Thiết bị tính toán: {device}')
if torch.cuda.is_available():
    print(f'🔥 GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM khả dụng: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ Đang sử dụng CPU! Khuyến nghị bật GPU T4/A100 trên Google Colab.')
print('=' * 65)


## 🟢 Bước 1: Xác định Bài toán & Tiêu chí Đánh giá (Define & Metrics)
- **Nhiệm vụ:** Phân loại biểu cảm khuôn mặt 6 lớp từ ảnh xám kích thước 48×48.
- **Bảng 6 nhãn cảm xúc chuẩn hóa:**
  1. `Contempt` (Khinh khỉnh / coi thường)
  2. `Fear` (Sợ hãi / lo lắng - Fear & Anxiety)
  3. `Happy` (Vui vẻ / tích cực)
  4. `Normal` (Bình thản / tự tin / trung tính)
  5. `Sad` (Buồn bã / chán nản)
  6. `Surprised` (Ngạc nhiên / bất ngờ)
- **Metric chấp nhận:** Overall Validation Accuracy $\ge 70\%$, F1 $\ge 0.60$ cho các lớp khó, Inference latency $< 50$ms.


In [ ]:
# Cấu hình các tham số thực nghiệm (Experiment Configuration)
CONFIG = {
    'emotion_classes': ['Contempt', 'Fear', 'Happy', 'Normal', 'Sad', 'Surprised'],
    'num_classes': 6,
    'input_size': 48,           # Ảnh xám 48x48 pixel
    'in_channels': 1,          # 1 kênh Grayscale
    'batch_size': 64,
    'epochs': 50,
    'learning_rate': 1e-3,     # AdamW initial LR
    'weight_decay': 1e-4,      # L2 Regularization
    'dropout_rate': 0.5,       # Chống overfit cho mạng from scratch
    'label_smoothing': 0.1,    # Chống overconfidence
    'patience': 8,             # Early Stopping patience
    'min_lr': 1e-6,
    'save_path': 'weights/custom_resnet18_6emo.pt',
    'history_path': 'train_history.json',
    'metrics_path': 'docs/evaluation_metrics.json'
}

EMOTION_CLASSES = CONFIG['emotion_classes']
print('Danh sách 6 nhãn cảm xúc:')
for idx, cls_name in enumerate(EMOTION_CLASSES):
    print(f'  [{idx}] {cls_name}')


## 🟢 Bước 2: Thu thập, Giải nén & Khám phá Dữ liệu (Data Engineering)
Tự động tìm kiếm file nén `Dataset.zip` trên Google Drive (hoặc ổ đĩa cục bộ), giải nén vào ổ SSD tốc độ cao của Colab (`/content/Data/Dataset`) và kiểm tra tính toàn vẹn của 6 nhãn cảm xúc.


In [ ]:
# Tự động phát hiện và giải nén Dataset.zip nếu dữ liệu chưa sẵn sàng
target_extract_dir = '/content/Data/Dataset' if IN_COLAB else 'Data/Dataset'
train_check_path = os.path.join(target_extract_dir, 'Train')

# Các vị trí tiềm năng của file zip
zip_candidates = [
    '/content/drive/MyDrive/emotion_recognition/Dataset.zip',
    '/content/drive/MyDrive/Dataset.zip',
    '/content/Dataset.zip',
    'Dataset.zip',
    'Data/Dataset.zip'
]

if not os.path.exists(train_check_path):
    found_zip = None
    for zpath in zip_candidates:
        if os.path.exists(zpath):
            found_zip = zpath
            break

    if found_zip:
        print(f'📦 Tìm thấy file zip tại: {found_zip}')
        print(f'⏳ Đang giải nén vào {target_extract_dir}...')
        os.makedirs(target_extract_dir, exist_ok=True)
        with zipfile.ZipFile(found_zip, 'r') as zip_ref:
            zip_ref.extractall(target_extract_dir)
        print('✅ Giải nén dữ liệu hoàn tất!')
    else:
        print('ℹ️ Chưa phát hiện file Dataset.zip, tiếp tục kiểm tra các thư mục dữ liệu sẵn có...')
else:
    print(f'✅ Thư mục dữ liệu đã tồn tại sẵn tại: {target_extract_dir}')


In [ ]:
# Tự động xác định đường dẫn thư mục DATA_DIR chuẩn
data_candidates = [
    '/content/Data/Dataset',
    'Data/Dataset',
    'Dataset',
    '/content/drive/MyDrive/emotion_recognition/Data/Dataset',
    '/content/drive/MyDrive/Data/Dataset'
]

DATA_DIR = None
for cand in data_candidates:
    if os.path.exists(os.path.join(cand, 'Train')) and os.path.exists(os.path.join(cand, 'Validation')):
        DATA_DIR = cand
        break

if DATA_DIR is None:
    # Nếu file giải nén có thêm 1 cấp thư mục Dataset/Dataset
    nested_candidate = '/content/Data/Dataset/Dataset'
    if os.path.exists(os.path.join(nested_candidate, 'Train')):
        DATA_DIR = nested_candidate
    else:
        DATA_DIR = target_extract_dir

print(f'📁 Thư mục dữ liệu được chọn: {DATA_DIR}')

# Kiểm tra số lượng ảnh và tính toàn vẹn
data_stats = {'Train': {}, 'Validation': {}}
corrupt_files = []

for split in ['Train', 'Validation']:
    split_path = os.path.join(DATA_DIR, split)
    for cls in EMOTION_CLASSES:
        cls_folder = os.path.join(split_path, cls)
        if os.path.exists(cls_folder):
            files = [f for f in os.listdir(cls_folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
            data_stats[split][cls] = len(files)
            # Kiểm tra mẫu 20 ảnh đầu tiên
            for f in files[:20]:
                try:
                    with Image.open(os.path.join(cls_folder, f)) as img:
                        img.verify()
                except Exception as e:
                    corrupt_files.append((os.path.join(cls_folder, f), str(e)))
        else:
            data_stats[split][cls] = 0

# In bảng thống kê dữ liệu
print('=' * 60)
print(f'{"Lớp (Emotion)":<15} | {"Train":<10} | {"Validation":<10} | {"Tổng":<10}')
print('-' * 60)
total_tr = sum(data_stats['Train'].values())
total_val = sum(data_stats['Validation'].values())
for cls in EMOTION_CLASSES:
    tr = data_stats['Train'].get(cls, 0)
    va = data_stats['Validation'].get(cls, 0)
    print(f'{cls:<15} | {tr:<10} | {va:<10} | {tr+va:<10}')
print('-' * 60)
print(f'{"TỔNG CỘNG":<15} | {total_tr:<10} | {total_val:<10} | {total_tr+total_val:<10}')
print('=' * 60)
if corrupt_files:
    print(f'⚠️ Phát hiện {len(corrupt_files)} file hỏng!')
else:
    print('✅ Tất cả mẫu ảnh kiểm tra đều nguyên vẹn 100%.')


In [ ]:
# Hiển thị ảnh mẫu đại diện cho 6 lớp cảm xúc (Ảnh xám Grayscale)
fig, axes = plt.subplots(len(EMOTION_CLASSES), 4, figsize=(10, 14))

for row_idx, cls in enumerate(EMOTION_CLASSES):
    cls_dir = os.path.join(DATA_DIR, 'Train', cls)
    if os.path.exists(cls_dir):
        sample_imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.png'))][:4]
    else:
        sample_imgs = []

    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        if col_idx < len(sample_imgs):
            img_path = os.path.join(cls_dir, sample_imgs[col_idx])
            img = Image.open(img_path).convert('L')
            ax.imshow(img, cmap='gray')
        ax.axis('off')
        if col_idx == 0:
            ax.set_title(f'{cls}', fontsize=12, fontweight='bold', loc='left')

plt.suptitle(f'Ảnh Mẫu 6 Nhãn Cảm Xúc ({DATA_DIR}/Train)', fontsize=14, y=0.92)
plt.tight_layout()
plt.show()


## 🟢 Bước 3: Tăng cường Dữ liệu & Xử lý Cân bằng (Augmentation & Balancing)
Khi huấn luyện mô hình CNN **from scratch** không có trọng số ImageNet, mô hình rất dễ bị học vẹt (overfit). Ta áp dụng chiến lược Data Augmentation đa chiều:
- **RandomCrop (54×54 -> 48×48):** Tạo khả năng bất biến với vị trí khuôn mặt.
- **RandomHorizontalFlip (p=0.5):** Tăng gấp đôi mẫu đối xứng.
- **RandomRotation (±10°):** Giúp nhận diện khi người nghiêng đầu.
- **ColorJitter:** Điều chỉnh sáng/tối mô phỏng các điều kiện ánh sáng camera/webcam.
- **RandomErasing:** Xóa ngẫu nhiên đốm nhỏ để mô hình không chỉ phụ thuộc vào một vùng đặc trưng khuôn mặt duy nhất.
- **Normalization:** Đưa dải điểm ảnh về $[-1, 1]$ với $\mu=0.5, \sigma=0.5$.


In [ ]:
# Thiết lập Data Transforms & Augmentation
input_size = CONFIG['input_size']
scale_size = int(input_size * 1.15)  # 55x55

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((scale_size, scale_size)),
    transforms.RandomCrop((input_size, input_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15), value=0.0)
])

val_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Nạp ImageFolder và tạo DataLoader
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'Train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'Validation'), transform=val_transforms)

num_workers = 2 if torch.cuda.is_available() else 0
train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=num_workers, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=num_workers, pin_memory=torch.cuda.is_available()
)

print(f'✅ Đã khởi tạo DataLoaders thành công!')
print(f'   - Số batches Train: {len(train_loader)} (Batch size: {CONFIG["batch_size"]})')
print(f'   - Số batches Validation: {len(val_loader)}')

# Tính toán Class Weights phòng ngừa mất cân bằng
train_counts = [data_stats['Train'].get(cls, 1) for cls in train_dataset.classes]
total_train_samples = sum(train_counts)
class_weights = [total_train_samples / (len(train_counts) * c) if c > 0 else 1.0 for c in train_counts]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print('\n⚖️  Trọng số cân bằng lớp (Class Weights):')
for cls, w in zip(train_dataset.classes, class_weights):
    print(f'   - {cls:<12}: {w:.4f}')


## 🟢 Bước 4: Thiết lập Kiến trúc CustomResNet18 from Scratch (Architecture Setup)
Mô hình ResNet-18 được lập trình **hoàn toàn từ đầu**, tùy chỉnh riêng cho ảnh 48×48 Grayscale:
1. **Tầng Stem (Conv1):** Dùng `kernel=3x3, stride=1, padding=1` (thay vì 7×7 stride 2 của ImageNet) để giữ nguyên độ phân giải không gian 48×48.
2. **Lược bỏ MaxPool đầu tiên:** Không làm giảm chiều quá sớm trên ảnh nhỏ.
3. **Residual Learning:** 4 nhóm tầng (mỗi nhóm gồm 2 khối `BasicBlock` với skip connection $F(x) + x$):
   - Layer 1: 64 channels ($48\times 48$)
   - Layer 2: 128 channels ($24\times 24$, stride=2)
   - Layer 3: 256 channels ($12\times 12$, stride=2)
   - Layer 4: 512 channels ($6\times 6$, stride=2)
4. **Global Pooling & Dropout:** `AdaptiveAvgPool2d((1, 1))` + `Dropout(0.5)` + `Linear(512, 6)`.
5. **Khởi tạo Kaiming He Normal:** Tối ưu hóa gradient flow cho hàm kích hoạt ReLU.


In [ ]:
# 1. Khối BasicBlock với Skip Connection
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity  # Skip connection
        out = self.relu(out)
        return out

# 2. Mạng CustomResNet18
class CustomResNet18(nn.Module):
    def __init__(self, num_classes=6, in_channels=1, dropout_rate=0.5):
        super(CustomResNet18, self).__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels

        # Tầng Stem cho ảnh 48x48
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        # 4 Nhóm tầng Residual
        self.layer1 = self._make_layer(64, 64, num_blocks=2, stride=1)
        self.layer2 = self._make_layer(64, 128, num_blocks=2, stride=2)   # 48 -> 24
        self.layer3 = self._make_layer(128, 256, num_blocks=2, stride=2)  # 24 -> 12
        self.layer4 = self._make_layer(256, 512, num_blocks=2, stride=2)  # 12 -> 6

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(512 * BasicBlock.expansion, num_classes)

        # Khởi tạo trọng số Kaiming He
        self._initialize_weights()

    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        downsample = None
        if stride != 1 or in_channels != out_channels * BasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * BasicBlock.expansion)
            )
        layers = []
        layers.append(BasicBlock(in_channels, out_channels, stride, downsample))
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(out_channels * BasicBlock.expansion, out_channels))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

    def predict(self, x):
        scores = self.forward(x)
        _, predictions = scores.max(1)
        return predictions

    def predict_proba(self, x):
        scores = self.forward(x)
        probs = F.softmax(scores, dim=1)
        max_probs, predictions = probs.max(1)
        return max_probs, predictions

# Khởi tạo mô hình và kiểm tra cấu trúc
model = CustomResNet18(
    num_classes=CONFIG['num_classes'],
    in_channels=CONFIG['in_channels'],
    dropout_rate=CONFIG['dropout_rate']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'🏛️ Mô hình: CustomResNet18 (From Scratch)')
print(f'   - Tổng số tham số:      {total_params:,}')
print(f'   - Tham số huấn luyện:  {trainable_params:,}')

# Thử nghiệm forward pass với dummy batch [4, 1, 48, 48]
dummy = torch.randn(4, 1, 48, 48).to(device)
with torch.no_grad():
    out = model(dummy)
assert out.shape == (4, 6), f'Shape không đúng: {out.shape}'
print(f'✅ Forward pass test thành công! Shape đầu ra: {out.shape}')


## 🟢 Bước 5: Huấn luyện & Giám sát Thực nghiệm (Training & Monitoring)
Chiến lược huấn luyện 1 giai đoạn end-to-end:
- **Hàm mất mát:** `CrossEntropyLoss` kết hợp `class_weights` và `label_smoothing=0.1` (giảm overconfidence).
- **Tối ưu hóa:** `AdamW` (lr=1e-3, weight_decay=1e-4) tìm kiếm cực tiểu phẳng (Flat Minimum).
- **Lịch điều chỉnh tốc độ học:** `CosineAnnealingLR` hạ cánh nhẹ nhàng về $\eta_{min}=1e-6$.
- **Gradient Clipping (5.0):** Ổn định vector gradient.
- **Early Stopping (patience=8):** Tự ngắt khi validation loss không còn cải thiện để chống học vẹt.
- **Checkpoint:** Chỉ lưu giữ bộ trọng số đạt **Validation Accuracy cao nhất**.


In [ ]:
# Thiết lập Loss Function, Optimizer, Scheduler
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=CONFIG['label_smoothing']
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['epochs'],
    eta_min=CONFIG['min_lr']
)

# Các hàm huấn luyện từng epoch
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total


In [ ]:
# Vòng lặp Huấn luyện (Training Loop với Early Stopping & Best Checkpoint)
os.makedirs(os.path.dirname(CONFIG['save_path']), exist_ok=True)

best_val_acc = 0.0
best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'learning_rates': []
}

print('=' * 75)
print(f'🚀 BẮT ĐẦU HUẤN LUYỆN CUSTOMRESNET18 — TỐI ĐA {CONFIG["epochs"]} EPOCHS')
print('=' * 75)

for epoch in range(1, CONFIG['epochs'] + 1):
    t0 = time.time()
    current_lr = optimizer.param_groups[0]['lr']

    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    elapsed = time.time() - t0

    history['train_loss'].append(round(tr_loss, 4))
    history['train_acc'].append(round(tr_acc, 4))
    history['val_loss'].append(round(val_loss, 4))
    history['val_acc'].append(round(val_acc, 4))
    history['learning_rates'].append(current_lr)

    print(
        f'Epoch [{epoch:02d}/{CONFIG["epochs"]:02d}] ({elapsed:.1f}s, lr={current_lr:.1e}) | '
        f'Train Loss: {tr_loss:.4f} - Acc: {tr_acc * 100:.2f}% | '
        f'Val Loss: {val_loss:.4f} - Acc: {val_acc * 100:.2f}%'
    )

    # Lưu checkpoint tốt nhất
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), CONFIG['save_path'])
        print(f'  ⭐ [LƯU BEST MODEL] Kỷ lục mới: Val Acc = {val_acc * 100:.2f}% (Loss: {val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f'\n⏹️ [EARLY STOPPING] Kích hoạt tại Epoch {epoch} sau {CONFIG["patience"]} epochs không cải thiện.')
            break

# Lưu file lịch sử huấn luyện
with open(CONFIG['history_path'], 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

print('=' * 75)
print(f'🎉 HUẤN LUYỆN HOÀN TẤT!')
print(f'   - Best Validation Accuracy: {best_val_acc * 100:.2f}% (Epoch {best_epoch})')
print(f'   - File trọng số đã lưu:     {CONFIG["save_path"]}')
print('=' * 75)


## 📊 Trực quan hóa Đường cong Huấn luyện (Learning Curves)
Phân tích đồ thị Loss và Accuracy để đánh giá khả năng hội tụ và khoảng cách Overfitting Gap.


In [ ]:
# Vẽ đồ thị Loss & Accuracy
epochs_range = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Đồ thị Loss
ax1.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 'r--', label='Validation Loss', linewidth=2)
ax1.set_title('Cross-Entropy Loss qua từng Epoch', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Đồ thị Accuracy
ax2.plot(epochs_range, [x * 100 for x in history['train_acc']], 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs_range, [x * 100 for x in history['val_acc']], 'r--', label='Validation Accuracy', linewidth=2)
ax2.axhline(y=70, color='g', linestyle=':', label='Mục tiêu (70%)')
ax2.set_title('Độ chính xác (Accuracy %) qua từng Epoch', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('docs', exist_ok=True)
plt.savefig('docs/training_curves.png', dpi=200)
plt.show()


## 🟢 Bước 6: Đánh giá Độc lập & Ma trận Nhầm lẫn (Evaluation & Confusion Matrix)
Nạp bộ trọng số tối ưu nhất (`best_model`) để thi đấu thật trên toàn bộ 3,000 ảnh tập Validation:
- Tính toán **Ma trận nhầm lẫn (Confusion Matrix 6×6)**.
- Đo lường **Precision**, **Recall**, **F1-Score** cho từng nhãn cảm xúc.
- Phân tích sâu các cặp cảm xúc dễ gây nhầm lẫn nhất (ví dụ: `Fear` vs `Sad`, `Contempt` vs `Normal`).


In [ ]:
# Nạp lại trọng số Best Checkpoint
best_model = CustomResNet18(
    num_classes=CONFIG['num_classes'],
    in_channels=CONFIG['in_channels'],
    dropout_rate=CONFIG['dropout_rate']
).to(device)

best_model.load_state_dict(torch.load(CONFIG['save_path'], map_location=device))
best_model.eval()

# Thu thập dự đoán trên tập Validation
all_preds = []
all_targets = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = best_model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Tính toán Confusion Matrix 6x6
num_classes = len(EMOTION_CLASSES)
cm = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(all_targets, all_preds):
    cm[t, p] += 1

# Tính Precision, Recall, F1
metrics = {}
total_samples = len(all_targets)
correct_samples = np.trace(cm)
overall_accuracy = correct_samples / total_samples

print('=' * 75)
print('📈 BẢNG ĐÁNH GIÁ CHỈ SỐ CHI TIẾT THEO TỪNG NHÃN CẢM XÚC:')
print(f'{"Lớp (Emotion)":<15} | {"Precision":<10} | {"Recall":<10} | {"F1-Score":<10} | {"Support":<10}')
print('-' * 75)

for i, cls in enumerate(EMOTION_CLASSES):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    support = cm[i, :].sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    metrics[cls] = {'precision': round(prec, 4), 'recall': round(rec, 4), 'f1_score': round(f1, 4), 'support': int(support)}
    print(f'{cls:<15} | {prec * 100:>8.2f}% | {rec * 100:>8.2f}% | {f1:>8.4f}  | {support:>10}')

print('-' * 75)
print(f'{"ĐỘ CHÍNH XÁC CHUNG (ACCURACY)":<35} : {overall_accuracy * 100:.2f}%')
print('=' * 75)

metrics['overall_accuracy'] = round(float(overall_accuracy), 4)
with open(CONFIG['metrics_path'], 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)


In [ ]:
# Vẽ Ma trận Nhầm lẫn (Confusion Matrix Heatmap 6x6)
row_sums = cm.sum(axis=1)[:, np.newaxis]
cm_norm = np.divide(cm.astype('float'), row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(num_classes),
    yticks=np.arange(num_classes),
    xticklabels=EMOTION_CLASSES,
    yticklabels=EMOTION_CLASSES,
    title=f'Confusion Matrix 6 Cảm Xúc (Độ chính xác: {overall_accuracy * 100:.2f}%)',
    ylabel='Nhãn Thực Tế (True Label)',
    xlabel='Nhãn Dự Đoán (Predicted Label)'
)
plt.setp(ax.get_xticklabels(), rotation=35, ha='right', rotation_mode='anchor')

thresh = cm_norm.max() / 2.
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(
            j, i, f'{cm_norm[i, j]*100:.1f}%\n({cm[i, j]})',
            ha='center', va='center',
            color='white' if cm_norm[i, j] > thresh else 'black',
            fontsize=9
        )

plt.tight_layout()
plt.savefig('docs/confusion_matrix.png', dpi=200)
plt.show()


## 🟢 Bước 7: Tối ưu hóa & Đóng gói Triển khai (Optimization & ONNX Export)
- **Export ONNX:** Chuyển đổi mô hình sang định dạng chuẩn mở ONNX với `dynamic_axes` hỗ trợ batch động, tăng tốc độ inference trên máy chủ CPU/Web lên gấp 2-4 lần.
- **Kiểm tra độ trễ (Inference Latency):** Đo đạc thời gian dự đoán 1 bức ảnh nhằm đảm bảo $< 50$ms cho ứng dụng phỏng vấn thời gian thực.
- **Dự đoán thử nghiệm trên ảnh ngẫu nhiên.**


In [ ]:
# 1. Export sang định dạng ONNX
onnx_path = 'weights/custom_resnet18_6emo.onnx'
dummy_input = torch.randn(1, 1, 48, 48).to(device)

torch.onnx.export(
    best_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input_face'],
    output_names=['emotion_logits'],
    dynamic_axes={'input_face': {0: 'batch_size'}, 'emotion_logits': {0: 'batch_size'}}
)

print(f'📦 Đã xuất mô hình ONNX thành công tại: {onnx_path}')
print(f'   Dung lượng file ONNX: {os.path.getsize(onnx_path) / 1e6:.2f} MB')

# 2. Đo đếm Inference Latency (ms / ảnh)
best_model.eval()
times = []
test_tensor = torch.randn(1, 1, 48, 48).to(device)

# Warm-up 10 lần
for _ in range(10):
    with torch.no_grad():
        _ = best_model(test_tensor)

# Đo 100 lần
for _ in range(100):
    t_start = time.perf_counter()
    with torch.no_grad():
        _ = best_model(test_tensor)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times.append((time.perf_counter() - t_start) * 1000)

avg_latency = np.mean(times)
p95_latency = np.percentile(times, 95)
print(f'⚡ Tốc độ suy luận (Inference Latency):')
print(f'   - Trung bình: {avg_latency:.2f} ms / ảnh')
print(f'   - 95th Percentile: {p95_latency:.2f} ms / ảnh')
if avg_latency < 50.0:
    print('   ✅ ĐẠT TIÊU CHUẨN REAL-TIME CHO ỨNG DỤNG WEB (< 50ms)!')


In [ ]:
# 3. Thử nghiệm dự đoán trực quan trên một số ảnh kiểm thử
val_subset = [val_dataset[i] for i in random.sample(range(len(val_dataset)), 6)]

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
best_model.eval()

for idx, (img_tensor, true_label) in enumerate(val_subset):
    input_t = img_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        prob, pred = best_model.predict_proba(input_t)

    pred_cls = EMOTION_CLASSES[pred.item()]
    true_cls = EMOTION_CLASSES[true_label]
    conf = prob.item() * 100

    ax = axes[idx]
    # Denormalize để hiển thị
    img_disp = img_tensor.squeeze().cpu().numpy() * 0.5 + 0.5
    ax.imshow(img_disp, cmap='gray')
    ax.axis('off')
    color = 'green' if pred_cls == true_cls else 'red'
    ax.set_title(f'Pred: {pred_cls}\n({conf:.1f}%)\nTrue: {true_cls}', fontsize=10, color=color)

plt.suptitle('Dự đoán thực tế trên tập Validation', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()


## 🏁 Tổng kết Thực nghiệm & Hướng dẫn Sử dụng
1. **Mô hình huấn luyện:** Đã hoàn thành huấn luyện `CustomResNet18` from scratch 6 nhãn cảm xúc.
2. **Files đầu ra:**
   - Trọng số PyTorch: `weights/custom_resnet18_6emo.pt`
   - Mô hình ONNX: `weights/custom_resnet18_6emo.onnx`
   - Lịch sử huấn luyện: `train_history.json`
   - Bảng chỉ số đánh giá: `docs/evaluation_metrics.json`
   - Biểu đồ: `docs/training_curves.png`, `docs/confusion_matrix.png`
3. **Tích hợp vào backend:** File trọng số mới sẵn sàng được nạp trực tiếp qua `model.py` hoặc `crowd_emotion_model.py`.
